In [ ]:
USE database_name.schema_name;

In [ ]:
CREATE OR REPLACE FILE FORMAT schema_name.csv_ff
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF = ('', 'NULL');

In [ ]:
CREATE OR REPLACE STAGE schema_name.ORDERDATA
    URL='azure://database_names.blob.core.windows.net/df02mdp'
    CREDENTIALS=(AZURE_SAS_TOKEN='sv=2024-11-04&ss=b&srt=sco&sp=rl&se=2026-04-28T18:10:24Z&st=2026-01-12T10:55:24Z&spr=https,http&sig=GoG%2FcdcI0pILGKolyOpaXfQvW7ogic6ewGQamT1dems%3D')
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'order data MM';

In [ ]:
CREATE TABLE IF NOT EXISTS schema_name.bronze_orders (
  idx                 NUMBER,
  order_id            STRING,
  restaurant_id       STRING,
  rider_id            STRING,
  customer_id         STRING,
  order_total         NUMBER(10,2),
  discount            NUMBER(10,2),
  state               STRING,
  order_time          TIMESTAMP_NTZ,
  pickup_time         TIMESTAMP_NTZ,
  delivery_time       TIMESTAMP_NTZ,
  delivery_latitude   FLOAT,
  delivery_longitude  FLOAT,
  rider_vehicle       STRING,
  rider_payment       STRING,
  rider_tip           NUMBER(10,2),
  app_version         STRING,
  device_type         STRING,
  file_name           STRING,
  load_ts             TIMESTAMP_NTZ
);


In [ ]:
COPY INTO schema_name.bronze_orders (
  idx, order_id, restaurant_id, rider_id, customer_id,
  order_total, discount, state,
  order_time, pickup_time, delivery_time,
  delivery_latitude, delivery_longitude,
  rider_vehicle, rider_payment, rider_tip,
  app_version, device_type,
  file_name, load_ts
)
FROM (
  SELECT
    t.$1::NUMBER                                 AS idx,
    t.$2::STRING                                 AS order_id,
    t.$3::STRING                                 AS restaurant_id,
    t.$4::STRING                                 AS rider_id,
    t.$5::STRING                                 AS customer_id,
    t.$6::NUMBER(10,2)                           AS order_total,
    t.$7::NUMBER(10,2)                           AS discount,
    t.$8::STRING                                 AS state,
    t.$9::TIMESTAMP_NTZ                          AS order_time,
    t.$10::TIMESTAMP_NTZ                         AS pickup_time,
    t.$11::TIMESTAMP_NTZ                         AS delivery_time,
    t.$12::FLOAT                                 AS delivery_latitude,
    t.$13::FLOAT                                 AS delivery_longitude,
    t.$14::STRING                                AS rider_vehicle,
    t.$15::STRING                                AS rider_payment,
    t.$16::NUMBER(10,2)                          AS rider_tip,
    t.$17::STRING                                AS app_version,
    t.$18::STRING                                AS device_type,
    METADATA$FILENAME                            AS file_name,
    CURRENT_TIMESTAMP()                          AS load_ts
  FROM @schema_name.orderdata (FILE_FORMAT => schema_name.csv_ff) AS t
)
ON_ERROR = 'CONTINUE';
-- Notes:
-- * FORCE defaults to FALSE, so Snowflake will skip files it already loaded.
-- * Add PATTERN='.*\.csv' on the stage reference if your stage contains mixed file types.

In [ ]:
UPDATE status_flags SET datainflag = 0;